# Final Project

## Introduction
*What dataset are you looking at? Where/how was it created? What questions will be asked?*

Bu projede California Bölgesi Konut Fiyatları (Housing) veri setini inceliyoruz. Veri seti, farklı bölgelerdeki medyan konut değerleri, kişi sayısı, oda sayısı gibi mekânsal ve demografik özellikleri barındırmaktadır.

**Araştırma Soruları:**
1. Medyan konut fiyatları ile bölgelerin okyanusa yakınlığı arasında nasıl bir ilişki vardır?
2. Hane halkı geliri, ev fiyatlarını nasıl etkiler?
3. En pahalı evler nerelerde konumlanmaktadır?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/housing.csv')
df.head()

## Cleaning and Organizing Data

Veri seti üzerinde öncelikle bir teşhis (diagnostics) işlemi yapıyoruz. Eğik değerleri ve eksik (missing) verileri inceliyoruz. Eksik verilerin dağılımını bir ısı haritası (heatmap) ile görselleştirip ardından doldurma (imputation) işlemleri ile devam edeceğiz. Ayrıca bazı aykırı değerleri (outliers) inceleyip bunları projeye uygun şekilde yöneteceğiz.

In [ ]:
# 1. Eksik verilerin görselleştirilmesi
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Eksik Değerlerin Isı Haritası")
plt.show()

# Sayısal bir özet alalım:
missing_info = df.isnull().sum()
print("Eksik Veri Sayıları:\n", missing_info[missing_info > 0])

# `total_bedrooms` değişkeninde eksik veriler var. Bu eksik değerleri aynı mahallenin özelliklerini 
# çok değiştirmemek adına, medyan değer kullanarak dolduracağız (imputation).
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# 2. Aykırı değer (Outlier) analizi
# California housing verisinde `median_house_value` sıklıkla 500,001 değerinde kesilir (capped).
# Bu durumu analiz edelim:
capped_houses = df[df['median_house_value'] >= 500000]
print(f"\n500,000 $ ve üzeri (sınırlandırılmış) ev verisi sayısı: {len(capped_houses)}")

# Analizin doğruluğu ve modellemeler için, bu sınırlandırılmış (capped) verilerin bazen çıkartılması tercih edilir.
# Veri setimiz yeterince büyük olduğundan, kesilmiş olan (capped) maksimum fiyatlı değerleri ve maksimum hane gelirlerini (capped at 15.0001) filtreleyerek
# suni outlier'ları engelleyebiliriz.
df_clean = df[df['median_house_value'] < 500000].copy()

print(f"\nTemizlik öncesi satır sayısı: {len(df)}")
print(f"Temizlik sonrası (outliers/capped removed) satır sayısı: {len(df_clean)}")

## Visualizations

Bu bölümde konut fiyatlarına, coğrafi dağılıma ve bağımsız değişkenlerin özelliklerine odaklanan **4 farklı görselleştirme** yapacağız.

1. **Ev Fiyatlarının Sıklık Dağılımı:** Fiyatların nasıl bir dağılım sergilediği incelenecek.
2. **Okyanusa Yakınlığa Göre Medyan Fiyatlar:** Kategorik değişken üzerinden fiyat dağılımları (Boxplot).
3. **Gelir Düzeyinin Ev Fiyatlarına Etkisi:** Hane geliri ile ev fiyatları arasında nasıl bir doğrusal (linear) bir ilişki olduğu (Scatterplot).
4. **Coğrafi Fiyat Isı Haritası:** Koordinat sistemine göre nerede en pahalı / en yoğun evlerin bulunduğunu ortaya koyan bir haritalandırma tekniği.

In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('California Housing Data - Exploratory Visualizations', fontsize=18)

# 1. Histogram of Median House Values
sns.histplot(df_clean['median_house_value'], kde=True, bins=50, ax=axes[0,0], color='skyblue')
axes[0,0].set_title("1. Median House Value Dağılımı")
axes[0,0].set_xlabel("Medyan Ev Fiyatı ($)")
axes[0,0].set_ylabel("Frekans")

# 2. Boxplot: Ocean Proximity vs. House Value
sns.boxplot(x='ocean_proximity', y='median_house_value', data=df_clean, ax=axes[0,1], palette='Pastel1')
axes[0,1].set_title("2. Okyanusa Yakınlığa Göre Ev Fiyatları")
axes[0,1].set_xlabel("Okyanusa Yakınlık")
axes[0,1].set_ylabel("Medyan Ev Fiyatı ($)")

# 3. Scatter Plot: Median Income vs. House Value
sns.scatterplot(x='median_income', y='median_house_value', data=df_clean, alpha=0.3, ax=axes[1,0], color='coral')
axes[1,0].set_title("3. Hane Gelirinin Ev Fiyatlarına Etkisi")
axes[1,0].set_xlabel("Medyan Gelir (On Bin $)")
axes[1,0].set_ylabel("Medyan Ev Fiyatı ($)")

# 4. Geographical Data (Longitude vs Latitude colored by House Value)
scatter = axes[1,1].scatter(df_clean['longitude'], df_clean['latitude'], 
                            c=df_clean['median_house_value'], s=df_clean['population']/100, 
                            cmap='jet', alpha=0.5)
axes[1,1].set_title("4. Coğrafi Fiyat Dağılımı ve Nüfus Yoğunluğu (Daire Boyutu: Nüfus)")
axes[1,1].set_xlabel("Boylam (Longitude)")
axes[1,1].set_ylabel("Enlem (Latitude)")
plt.colorbar(scatter, ax=axes[1,1], label='Medyan Ev Fiyatı ($)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Görselleştirmelerin Yorumları:**
1. **Histogram:** Kesilmiş (capped) değerler çıkarıldıktan sonra, fiyatların çoğunlukla 100.000 ile 300.000 $ civarında yoğunlaştığı ve sağa çarpık (right-skewed) bir yapı sergilediği görülmektedir.
2. **Boxplot:** `INLAND` (iç kesimler) en düşük ev fiyatlarına sahipken, `<1H OCEAN` (okyanusa yakın) ve `NEAR OCEAN` evlerin değerlerinin önemli ölçüde arttığı gözlemlenmektedir. Özellikle ada evleri (ISLAND), en yüksek medyan fiyat değerlerine sahiptir.
3. **Scatter Plot:** Gelir düzeyi arttıkça medyan ev fiyatlarının da artış eğiliminde olduğu belirgin şekilde tespit edilebilir. Bu özellik, pozitif güçlü bir doğrusal ilişkinin varlığını akla getirmektedir.
4. **Coğrafi Grafik:** California kordon boyuna, yani sahil hattına paralellik arz eden kısımlarda, özellikle Los Angeles ve San Francisco (ve Bay Area) bölgesindeki evlerin hem daha pahalı olduğu hem de buraların daha fazla nüfusa (büyük noktalar) sahip olduğu görülmektedir.

## Descriptive Analysis

Betimsel analiz kısmında değişkenlerin birbiri ile olan ilişkilerini sayısal bir biçimde anlamlandırmaya odaklanacağız.

1. **İstatistiksel Özet Tablosu & Gruplama (Aggregation):** Ev fiyatlarının ve gelir durumunun Okyanusa yakınlık faktörüne göre ortalama medyan değerleri alınacak.
2. **Korelasyon Matrisi (Heatmap):** Sayısal veriler arasında doğrusal ilişkiler incelenip yorumlanacak. (Özellikle medyan ev fiyatı ile olan ilişkiler).

In [ ]:
# 1. Aggregation / Groupby
summary_stats = df_clean.groupby('ocean_proximity')[['median_house_value', 'median_income']].mean().round(2).sort_values(by='median_house_value', ascending=False)
display(summary_stats)

print("\nYorum: Ada (ISLAND) ve okyanus çevresindeki mahallelerin hem gelir düzeyi yüksek hem de ev fiyatları ortalaması bariz şekilde daha çoktur.\nİç bölgelerde (INLAND) yaşayanların ortalama konut fiyatı okyanusa yakın olanların neredeyse yarısı kadardır.\n")

# 2. Korelasyon Matrisi (Correlation Matrix)
numeric_df = df_clean.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Sayısal Değişkenler Arası Korelasyon Matrisi")
plt.show()

print("Korelasyon Yorumu:")
print("- `median_income` ile `median_house_value` arasında pozitif, görece yüksek bir korelasyon (r=~0.64) mevcuttur.")
print("- `total_rooms` ile `total_bedrooms`, ya da `population` ile `households` arasına kendi aralarında neredeyse 1'e yakın (0.91+) çok güçlü doğrusal ilişkiler vardır, bu durum multicollinearity'e işaret eder.")

## Conclusions

Bu çalışmada, **California Housing** veri seti üzerinde keşfedici bir veri analizi (Exploratory Data Analysis - EDA) yapılmıştır. Analizimizle birlikte proje başında sorduğumuz araştırma soruları yanıtlanmıştır:

1. **Denize Yakınlığın Fiyatlara Etkisi:** Okyanusa yaklaştıkça veya sahil kesimine doğru indikçe medyan ev fiyatları artış göstermektedir. Bu veri içerisindeki Ada (Island) özellikli yerleşimler en pahalı evlere ev sahipliği yapsa da sayısal olarak kısıtlıdır. Gerçek metropol yaşantısında San Francisco gibi körfez/kıyı boyu bölgelerde büyük yoğunluk mevcuttur. İç bölgelerde ev fiyatları en düşük seviyesine ulaşmıştır.
2. **Gelir & Fiyat Doğrusallığı:** Tahmin edileceği üzere evin fiyatını belirleyen en önemli etkenlerden ve belirleyicilerden biri hane halkının medyan geliridir (`median_income` ile yapılan pozitif korelasyon verisi bunu kanıtlar).
3. **Coğrafi Konum:** Haritalama üzerine çıkarılan görselleştirmeden net biçimde anlaşıldığı üzere California'nın sahil hatlarındaki Los Angeles civarı vb. yüksek nüfus yoğunluklu semtlerindeki konutlar açık arayla eyaletin en bahalı birimleridir.

**Eksikler & Sınırlandırmalar (Limitations):**
* Veri setinde `500.001 $` tavanıyla fiyatlar "cap" (sınır) sistemine takıldığı için 500 bin dolardan daha pahalı evler doğru şekilde ölçülememiştir. Analizin çarpılmaması için biz o kısmı temizleyerek ilerledik.
* Bu analiz ileride yapılacak bir Makine Öğrenmesi (Regression) modellemesi çalışmasına rehberlik edebilecek seviyede temel değişken ilişkilerini açıklamaktadır. İleride PCA ile çoklu doğrusal bağ (multicollinearity) problemleri düzeltilerek bir fiyat tahmin modeli geliştirilebilir.